In [47]:
import pandas as pd
import numpy as np
import joblib

In [48]:
model = joblib.load('../outputs/titanic_model.pkl')

In [49]:
df = pd.read_csv('../data/test.csv')

In [50]:
df.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [51]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  418 non-null    int64  
 1   Pclass       418 non-null    int64  
 2   Name         418 non-null    str    
 3   Sex          418 non-null    str    
 4   Age          332 non-null    float64
 5   SibSp        418 non-null    int64  
 6   Parch        418 non-null    int64  
 7   Ticket       418 non-null    str    
 8   Fare         417 non-null    float64
 9   Cabin        91 non-null     str    
 10  Embarked     418 non-null    str    
dtypes: float64(2), int64(4), str(5)
memory usage: 36.1 KB


In [52]:
df.describe(include='all')

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
count,418.000000,418.000000,418,418,332.000000,418.000000,418.000000,418,417.000000,91,418
unique,NaN,NaN,418,2,NaN,NaN,NaN,363,NaN,76,3
top,NaN,NaN,"Kelly, Mr. James",male,NaN,NaN,NaN,PC 17608,NaN,B57 B59 B63 B66,S
freq,NaN,NaN,1,266,NaN,NaN,NaN,5,NaN,3,270
mean,1100.500000,2.265550,NaN,NaN,30.272590,0.447368,0.392344,NaN,35.627188,NaN,NaN
std,120.810458,0.841838,NaN,NaN,14.181209,0.896760,0.981429,NaN,55.907576,NaN,NaN
min,892.000000,1.000000,NaN,NaN,0.170000,0.000000,0.000000,NaN,0.000000,NaN,NaN
25%,996.250000,1.000000,NaN,NaN,21.000000,0.000000,0.000000,NaN,7.895800,NaN,NaN
50%,1100.500000,3.000000,NaN,NaN,27.000000,0.000000,0.000000,NaN,14.454200,NaN,NaN
75%,1204.750000,3.000000,NaN,NaN,39.000000,1.000000,0.000000,NaN,31.500000,NaN,NaN


In [53]:
df.isnull().sum()

PassengerId      0
Pclass           0
Name             0
Sex              0
Age             86
SibSp            0
Parch            0
Ticket           0
Fare             1
Cabin          327
Embarked         0
dtype: int64

In [54]:
conditions = [
    (df['Age'] < 12),
    (df['Age'] >= 12) & (df['Age'] < 18),
    (df['Age'] >= 18) & (df['Age'] < 60),
    (df['Age'] >= 60)
]

group = ['kid', 'teen', 'adult', 'elderly']

df['AgeGroup'] = np.select(conditions, group, default='unknown')

In [55]:
df['NameTitle'] = df['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)

In [56]:
mask = df['AgeGroup'] == 'unknown'

df.loc[mask, 'NameTitle'].value_counts()

NameTitle
Mr        57
Miss      14
Mrs       10
Master     4
Ms         1
Name: count, dtype: int64

In [57]:
TitleAge = {"Mr": "adult", "Mrs": "adult", "Dr": "adult", "Master": "kid"}

for title, age in TitleAge.items():

    mask = (df['AgeGroup'] == "unknown") & (df['NameTitle'] == title)

    df.loc[mask, 'AgeGroup'] = age

In [58]:
mask = (df['NameTitle'] == 'Ms') & (df['AgeGroup'] == 'unknown')

df.loc[mask, 'NameTitle'] = 'Miss'

In [59]:
df['TicketCount'] = df.groupby('Ticket').transform('size')

In [60]:
mask = (df['AgeGroup'] == "unknown") & (df['NameTitle'] == "Miss") & (df['TicketCount'] == 1)

df.loc[mask, 'AgeGroup'] = 'adult'

In [61]:
mask = (df['AgeGroup'] == "unknown") & (df['NameTitle'] == "Miss") & (df['Parch'] == 0)

df.loc[mask, 'AgeGroup'] = 'adult'

In [62]:
mask = (df['AgeGroup'] == "unknown") & (df['NameTitle'] == "Miss")

df.loc[mask, 'AgeGroup'] = 'kid'

In [63]:
PclassDeckLevel = {1: 3, 2: 2, 3: 1}

df['DeckLevel'] = df['Pclass'].map(PclassDeckLevel)


In [64]:
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

In [65]:
df = pd.get_dummies(df, columns=['AgeGroup'], drop_first=True)

df = df.rename(columns={
    'AgeGroup_elderly': 'IsElderly',
    'AgeGroup_kid': 'IsKid',
    'AgeGroup_teen': 'IsTeen'
})

In [66]:
mapage = {False: 0, True: 1}

df['IsElderly'] = df['IsElderly'].map(mapage)
df['IsKid'] = df['IsKid'].map(mapage)
df['IsTeen'] = df['IsTeen'].map(mapage)

In [67]:
columnsdel = [
    'PassengerId', 'Name', 'Age', 'Embarked', 
    'Cabin', 'NameTitle', 'SibSp', 'Parch', 
    'Ticket', 'TicketCount', 'Fare'
]

df = df.drop(columns=columnsdel)

In [68]:
mapsex = {"male": 0, "female": 1}

df['Sex'] = df['Sex'].map(mapsex)

df = df.rename(columns={'Sex': 'IsFemale'})

In [69]:
final_pred = model.predict(df)

In [70]:
test_df = pd.read_csv('../data/test.csv')

In [71]:
submission = pd.DataFrame({
    'PassengerId': test_df['PassengerId'],
    'Survived': final_pred
})

In [72]:
submission.to_csv('../outputs/submission_titanic.csv', index=False)